# Drug-label reconstruction (v3): **OCR-aware fine-tuning + text-accuracy evaluation** (Kaggle)

v3 starts from the v2 checkpoint (epoch 19, val masked PSNR 25.75 dB). Model, data split, losses and the colour
harmoniser are the same as in v2. It adds three OCR components:

| # | What | Where OCR is used |
|---|---|---|
| 1 | **Region-level text-accuracy metric.** Text lines are detected on the clean label, and only the lines that the damage mask touches are kept. Each line is cropped from every method's output and read with EasyOCR. The reading is compared with the reading of the same line on the clean label (CER, word accuracy, % of damaged words fixed). | evaluation (EasyOCR) |
| 2 | **OCR text loss during fine-tuning.** A frozen TrOCR recogniser reads the damaged lines of the predicted clean image x̂₀, using a differentiable `roi_align` crop. Its cross-entropy against the line's text is added to the v2 losses, at low-noise timesteps only. | training (TrOCR) |
| 3 | **OCR re-ranking at inference.** The sampler generates N DDIM samples with different seeds, and TrOCR keeps the one whose damaged text lines it reads most confidently. The text boxes come from the *damaged* input, so no ground truth is used. | inference (TrOCR) |

**Why the v2 OCR number (CER ~93–106 %) was not usable:** Tesseract read the *whole* 416×416 label in one pass, and
small text at that size is mostly noise to it. On 134 of 200 clean labels it read fewer than 5 characters. Only ~7 % of
each label is damaged, so whole-label CER mostly measured OCR noise, not reconstruction quality. v3 scores **only the
damaged lines**.

**To keep the evaluation fair,** the text loss and re-ranking use **TrOCR**, while the metric uses **EasyOCR**.
Training labels are lines where both engines agree on the clean image. The model therefore cannot score well simply by
fooling the OCR engine it is measured with.

**Inputs** (Add Input): the dataset (`clean/`, `damaged/`, `masks/`), `last_checkpoint.pth` and `dataset_split_v2.csv`.
Optionally also add `ocr_cache_v3.json` from an earlier v3 run, which skips the ~30–40 min OCR pre-pass.
Settings: **Internet ON** (downloads EasyOCR and TrOCR weights), **GPU T4 x2**, then **Save & Run All**.


In [1]:
# Cell 1: imports, global configuration and time guard (v3: OCR-aware fine-tuning, Kaggle)
import os, math, time, json, random, shutil, subprocess, warnings, copy, csv, gc
import psutil
from pathlib import Path

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity
from skimage.color import rgb2lab, deltaE_ciede2000

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)
NOTEBOOK_START = time.time()

QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"
INPUT_DIR = Path("/kaggle/input")

CFG = dict(
    image_size=416,
    crop_size=256,
    batch_size=16,
    base_channels=64,
    channel_mults=(1, 2, 3, 4),
    num_res_blocks=2,
    attn_levels=(3,),
    dropout=0.1,
    use_damaged_pixels=True,
    timesteps=1000,
    min_snr_gamma=5.0,
    mask_loss_weight=8.0,
    lambda_x0_l1=1.0,
    lambda_colour=1.0,
    lambda_edge=0.5,
    lr=2e-4,
    weight_decay=1e-4,
    warmup_steps=1000,
    ema_decay=0.999,
    grad_clip=1.0,
    resume_checkpoint=os.environ.get("RESUME_PATH", "/kaggle/input/datasets/ayushsinghh1086/last-checkpoint/last_checkpoint.pth"),
    split_csv=os.environ.get("SPLIT_CSV", "/kaggle/input/datasets/ayushsinghh1086/split-csv/dataset_split_v2.csv"),          # "auto" = search /kaggle/input for dataset_split_v2.csv, or give a path, or None
    SESSION_TRAIN_HOURS=9.0,
    resume_lr=1e-4,
    resume_warmup_steps=300,
    min_lr=1e-6,
    ckpt_every_min=30,
    NOTEBOOK_HOURS_LIMIT=10.8, # whole notebook stays under ~11 h (Kaggle hard limit is 12 h)
    # -------------------------------------------------
    # ---------------- OCR settings ----------------
    trocr_name="microsoft/trocr-small-printed",   # differentiable reader for the text loss + re-ranking
    ocr_cache="auto",          # "auto" = look for ocr_cache_v3.json in /kaggle/input, else build it (~30-40 min)
    ocr_cache_max_hours=1.0,
    ocr_mag_ratio=1.5,         # EasyOCR detector magnification (small text at 416 px)
    ocr_min_conf=0.4,          # EasyOCR confidence needed to keep a line
    consensus_cer=0.25,        # EasyOCR vs TrOCR disagreement allowed for a line to become a training label
    box_min_damage=0.05,       # a line counts as "damaged" if >= 5 % of its box is under the mask
    text_max_len=32,
    text_k_per_img=4,          # damaged lines per training crop
    text_max_boxes=8,          # lines per batch sent through TrOCR
    text_min_abar=0.5,         # text loss only when alpha_bar(t) >= 0.5 (x0 is meaningful)
    lambda_text=0.02,
    FT_HOURS=6.5,              # auto-shortened if the OCR cache has to be rebuilt
    ram_stop_percent=88,       # stop fine-tuning early (and still evaluate) if CPU RAM use passes this
    ft_lr=3e-5, ft_warmup_steps=200,
    ft_val_every_min=45,
    max_val_psnr_drop=0.3,     # fine-tuned model must stay within 0.3 dB masked PSNR of v2 to be selected
    eval_reserve_hours=2.0,    # time kept for the final test evaluation
    rerank_n=4,                # DDIM samples per label for OCR re-ranking
    ocr_val_images=48,
    ocr_test_images=300,
    harmonizer="lab_stats",    # selected on validation in v2
    # -------------------------------------------------
    ddim_steps=25,
    val_psnr_images=32,
    val_psnr_steps=10,
    val_loss_images=256,
    harmonizer_select_images=300,
    eval_max_test_images=None,
    eval_batch_size=16,
    ocr_eval_images=200,
    run_ocr_eval=True,
    seed=42,
    num_workers=2,             # fewer DataLoader workers = less CPU RAM
)
if QUICK_TEST:
    CFG.update(image_size=96, crop_size=64, batch_size=4, base_channels=32, channel_mults=(1, 2, 2, 2),
               num_res_blocks=1, SESSION_TRAIN_HOURS=0.01, ddim_steps=4, val_psnr_steps=3, val_psnr_images=4,
               val_loss_images=8, harmonizer_select_images=6, eval_max_test_images=10, eval_batch_size=4,
               ocr_eval_images=4, warmup_steps=5, resume_warmup_steps=3, num_workers=0, run_ocr_eval=False,
               ckpt_every_min=0.1, FT_HOURS=0.02, ocr_val_images=4, ocr_test_images=4, rerank_n=2,
               eval_reserve_hours=0.0, ft_val_every_min=0.5, text_max_boxes=4, ft_warmup_steps=2, ocr_cache_max_hours=0.2)

OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG["seed"] + 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
USE_AMP = device.type == "cuda"
torch.backends.cudnn.benchmark = True

def hours_used():
    return (time.time() - NOTEBOOK_START) / 3600.0

def hours_left():
    return CFG["NOTEBOOK_HOURS_LIMIT"] - hours_used()

def ram_percent():
    return psutil.virtual_memory().percent

print(f"Device: {device} | GPUs: {NUM_GPUS} | AMP: {USE_AMP} | outputs -> {OUT_DIR} | RAM {psutil.virtual_memory().total/2**30:.1f} GB ({ram_percent():.0f} % used)")
if NUM_GPUS:
    for i in range(NUM_GPUS):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

Device: cuda | GPUs: 2 | AMP: True | outputs -> /kaggle/working | RAM 31.3 GB (5 % used)
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [2]:
# Cell 2: find the dataset, the model checkpoint and the split CSV automatically
def find_one(patterns, what):
    for pat in patterns:
        hits = sorted(INPUT_DIR.rglob(pat))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"Could not find {what} in /kaggle/input. Check that it is attached.")

# dataset = the folder that contains clean/, damaged/ and masks/
DATA_ROOT = next((p.parent for p in sorted(INPUT_DIR.rglob("clean"))
                  if p.is_dir() and (p.parent / "damaged").is_dir() and (p.parent / "masks").is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError("Dataset folder with clean/, damaged/, masks/ not found")

RESUME_PATH = find_one(["best_ema.pth", "last_checkpoint.pth", "*.pth"], "the model checkpoint (.pth)")
SPLIT_CSV = find_one(["dataset_split_v2.csv", "*split*.csv"], "the split CSV")
CFG["resume_checkpoint"], CFG["split_csv"] = str(RESUME_PATH), str(SPLIT_CSV)

for sub in ["clean", "damaged", "masks"]:
    print(f"{sub:8s}: {len(list((DATA_ROOT / sub).iterdir()))} entries")
print("Dataset root:", DATA_ROOT)
print("Checkpoint  :", RESUME_PATH, f"({RESUME_PATH.stat().st_size/2**20:.0f} MB)")
print("Split CSV   :", SPLIT_CSV)

clean   : 6957 entries
damaged : 20526 entries
masks   : 20527 entries
Dataset root: /kaggle/input/datasets/ayushsinghh1086/dataset416/dataset416
Checkpoint  : /kaggle/input/datasets/ayushsinghh1086/416pathbestema/best_ema.pth (85 MB)
Split CSV   : /kaggle/input/datasets/ayushsinghh1086/dataset416-split-csv/dataset_split_v2.csv


In [3]:
# Cell 3: leakage-free dataset (same grouping, seed and ratios as v1) with damage-centred crops
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

def build_source_groups(root):
    clean_map, mask_map, groups = {}, {}, {}
    for f in os.listdir(root / "clean"):
        if not f.startswith(".") and Path(f).suffix.lower() in IMG_EXTS:
            clean_map[Path(f).stem.split("_")[0]] = f
    for f in os.listdir(root / "masks"):
        if not f.startswith(".") and Path(f).suffix.lower() in IMG_EXTS:
            mask_map[Path(f).stem] = f
    for f in os.listdir(root / "damaged"):
        if f.startswith(".") or Path(f).suffix.lower() not in IMG_EXTS:
            continue
        stem = Path(f).stem
        sid = stem.split("_")[0]
        if sid in clean_map and stem in mask_map:
            groups.setdefault(sid, []).append({"damaged": f, "clean": clean_map[sid], "mask": mask_map[stem]})
    return groups

def split_sources(groups, train_ratio=0.7, val_ratio=0.15, seed=42):
    sources = sorted(groups.keys())
    gen = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(sources), generator=gen).tolist()
    n_tr, n_va = int(len(sources) * train_ratio), int(len(sources) * val_ratio)
    tr = {sources[i] for i in idx[:n_tr]}
    va = {sources[i] for i in idx[n_tr:n_tr + n_va]}
    te = {sources[i] for i in idx[n_tr + n_va:]}
    assert tr.isdisjoint(va) and tr.isdisjoint(te) and va.isdisjoint(te), "Source leakage between splits!"
    return tr, va, te

def read_rgb(path, size):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"Could not read {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if img.shape[0] != size or img.shape[1] != size:
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32) / 255.0

def read_mask(path, size):
    m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if m is None:
        raise IOError(f"Could not read {path}")
    if m.shape[0] != size or m.shape[1] != size:
        m = cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST)
    return (m > 127).astype(np.float32)

class PharmacyLabelDataset(Dataset):
    # Returns (clean, damaged, mask) in [-1, 1] / {0, 1}. Train mode returns damage-centred random crops.
    def __init__(self, root, samples, image_size, crop_size=None, train=False):
        self.root, self.samples = Path(root), samples
        self.image_size, self.crop_size, self.train = image_size, crop_size, train

    def __len__(self):
        return len(self.samples)

    def _crop_window(self, mask):
        s, c = self.image_size, self.crop_size
        if c is None or c >= s:
            return 0, 0
        ys, xs = np.nonzero(mask)
        if len(ys) > 0 and random.random() < 0.85:
            k = random.randrange(len(ys))
            cy = ys[k] + random.randint(-c // 4, c // 4)
            cx = xs[k] + random.randint(-c // 4, c // 4)
            y0, x0 = int(np.clip(cy - c // 2, 0, s - c)), int(np.clip(cx - c // 2, 0, s - c))
        else:
            y0, x0 = random.randint(0, s - c), random.randint(0, s - c)
        return y0, x0

    def __getitem__(self, i):
        it = self.samples[i]
        clean = read_rgb(self.root / "clean" / it["clean"], self.image_size)
        damaged = read_rgb(self.root / "damaged" / it["damaged"], self.image_size)
        mask = read_mask(self.root / "masks" / it["mask"], self.image_size)
        if self.train and self.crop_size:
            y0, x0 = self._crop_window(mask)
            c = self.crop_size
            clean, damaged, mask = clean[y0:y0+c, x0:x0+c], damaged[y0:y0+c, x0:x0+c], mask[y0:y0+c, x0:x0+c]
        to_t = lambda a: torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1) * 2.0 - 1.0
        return to_t(clean), to_t(damaged), torch.from_numpy(np.ascontiguousarray(mask))[None]

def load_split_csv(path):
    # rows keep the original order, so test/val indices are identical across runs
    out = {"train": [], "val": [], "test": []}
    with open(path, newline="") as f:
        for r in csv.DictReader(f):
            out[r["split"]].append({"damaged": r["damaged_file"], "clean": r["clean_file"], "mask": r["mask_file"]})
    return out["train"], out["val"], out["test"]

groups = build_source_groups(DATA_ROOT)
if SPLIT_CSV is not None and SPLIT_CSV.exists():
    train_samples, val_samples, test_samples = load_split_csv(SPLIT_CSV)
    missing = [s for s in train_samples + val_samples + test_samples
               if not ((DATA_ROOT / "damaged" / s["damaged"]).exists() and (DATA_ROOT / "clean" / s["clean"]).exists()
                       and (DATA_ROOT / "masks" / s["mask"]).exists())]
    if missing:
        raise FileNotFoundError(f"{len(missing)} files listed in {SPLIT_CSV.name} are not in the dataset, e.g. {missing[0]}")
    sid = lambda s: s["damaged"].split("_")[0]
    train_src, val_src, test_src = ({sid(s) for s in ss} for ss in (train_samples, val_samples, test_samples))
    assert train_src.isdisjoint(val_src) and train_src.isdisjoint(test_src) and val_src.isdisjoint(test_src), "Source leakage!"
    # cross-check against the deterministic seed-42 split
    r_tr, r_va, r_te = split_sources(groups, seed=CFG["seed"])
    same = (r_tr, r_va, r_te) == (train_src, val_src, test_src)
    print(f"Split loaded from {SPLIT_CSV} | identical to the recomputed seed-{CFG['seed']} split: {same}")
else:
    print("No split CSV found -> recomputing the seed-42 split (same as the first run)")
    train_src, val_src, test_src = split_sources(groups, seed=CFG["seed"])
    flatten = lambda srcs: [s for sid in sorted(srcs) for s in groups[sid]]
    train_samples, val_samples, test_samples = flatten(train_src), flatten(val_src), flatten(test_src)

train_ds = PharmacyLabelDataset(DATA_ROOT, train_samples, CFG["image_size"], CFG["crop_size"], train=True)
val_ds = PharmacyLabelDataset(DATA_ROOT, val_samples, CFG["image_size"])
test_ds = PharmacyLabelDataset(DATA_ROOT, test_samples, CFG["image_size"])
print(f"Sources -> train {len(train_src)} | val {len(val_src)} | test {len(test_src)}")
print(f"Samples -> train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

# Save the split manifest for the report
with open(OUT_DIR / "dataset_split_v2.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["source_id", "clean_file", "damaged_file", "mask_file", "split"])
    for name, ss in [("train", train_samples), ("val", val_samples), ("test", test_samples)]:
        for s in ss:
            w.writerow([s["damaged"].split("_")[0], s["clean"], s["damaged"], s["mask"], name])

Split loaded from /kaggle/input/datasets/ayushsinghh1086/dataset416-split-csv/dataset_split_v2.csv | identical to the recomputed seed-42 split: True
Sources -> train 4869 | val 1043 | test 1045
Samples -> train 14362 | val 3088 | test 3073


In [4]:
# Cell 4: U-Net with per-block time conditioning and low-resolution self-attention
def timestep_embedding(t, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

def norm(ch):
    return nn.GroupNorm(32 if ch % 32 == 0 else 8, ch)

def zero_module(m):
    for p in m.parameters():
        nn.init.zeros_(p)
    return m

class ResBlock(nn.Module):
    def __init__(self, cin, cout, tdim, dropout):
        super().__init__()
        self.n1, self.c1 = norm(cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.t = nn.Linear(tdim, cout * 2)
        self.n2, self.drop = norm(cout), nn.Dropout(dropout)
        self.c2 = zero_module(nn.Conv2d(cout, cout, 3, padding=1))
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, temb):
        h = self.c1(F.silu(self.n1(x)))
        scale, shift = self.t(F.silu(temb))[:, :, None, None].chunk(2, dim=1)
        h = self.n2(h) * (1 + scale) + shift
        h = self.c2(self.drop(F.silu(h)))
        return h + self.skip(x)

class AttnBlock(nn.Module):
    def __init__(self, ch, heads=4):
        super().__init__()
        self.heads, self.n = heads, norm(ch)
        self.qkv = nn.Conv2d(ch, ch * 3, 1)
        self.proj = zero_module(nn.Conv2d(ch, ch, 1))

    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.qkv(self.n(x)).reshape(B, 3, self.heads, C // self.heads, H * W).unbind(1)
        q, k, v = (t.transpose(-1, -2) for t in (q, k, v))
        o = F.scaled_dot_product_attention(q, k, v)
        return x + self.proj(o.transpose(-1, -2).reshape(B, C, H, W))

class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x):
        return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode="nearest"))

class RestorationUNet(nn.Module):
    def __init__(self, in_ch=7, out_ch=3, base=64, mults=(1, 2, 3, 4), num_res=2, attn_levels=(3,), dropout=0.1):
        super().__init__()
        self.base, tdim = base, base * 4
        self.temb = nn.Sequential(nn.Linear(base, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.conv_in = nn.Conv2d(in_ch, base, 3, padding=1)
        self.down, chans, ch = nn.ModuleList(), [base], base
        for lvl, m in enumerate(mults):
            for _ in range(num_res):
                blk = nn.ModuleList([ResBlock(ch, base * m, tdim, dropout)]); ch = base * m
                if lvl in attn_levels:
                    blk.append(AttnBlock(ch))
                self.down.append(blk); chans.append(ch)
            if lvl != len(mults) - 1:
                self.down.append(nn.ModuleList([Downsample(ch)])); chans.append(ch)
        self.mid = nn.ModuleList([ResBlock(ch, ch, tdim, dropout), AttnBlock(ch), ResBlock(ch, ch, tdim, dropout)])
        self.up = nn.ModuleList()
        for lvl, m in reversed(list(enumerate(mults))):
            for i in range(num_res + 1):
                blk = nn.ModuleList([ResBlock(ch + chans.pop(), base * m, tdim, dropout)]); ch = base * m
                if lvl in attn_levels:
                    blk.append(AttnBlock(ch))
                if lvl != 0 and i == num_res:
                    blk.append(Upsample(ch))
                self.up.append(blk)
        self.out = nn.Sequential(norm(ch), nn.SiLU(), zero_module(nn.Conv2d(ch, out_ch, 3, padding=1)))
        self.depth = len(mults) - 1

    @staticmethod
    def _run(blk, h, temb):
        for layer in blk:
            h = layer(h, temb) if isinstance(layer, ResBlock) else layer(h)
        return h

    def forward(self, x, t):
        # pad so H and W are divisible by 2**depth, crop back at the end
        H, W = x.shape[-2:]
        f = 2 ** self.depth
        ph, pw = (f - H % f) % f, (f - W % f) % f
        if ph or pw:
            x = F.pad(x, (0, pw, 0, ph), mode="reflect")
        temb = self.temb(timestep_embedding(t, self.base))
        h = self.conv_in(x); hs = [h]
        for blk in self.down:
            h = self._run(blk, h, temb); hs.append(h)
        h = self._run(self.mid, h, temb)
        for blk in self.up:
            h = self._run(blk, torch.cat([h, hs.pop()], dim=1), temb)
        return self.out(h)[..., :H, :W]

model = RestorationUNet(in_ch=7, base=CFG["base_channels"], mults=CFG["channel_mults"],
                        num_res=CFG["num_res_blocks"], attn_levels=CFG["attn_levels"],
                        dropout=CFG["dropout"]).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"RestorationUNet parameters: {total_params/1e6:.2f}M")

RestorationUNet parameters: 22.29M


In [5]:
# Cell 5: cosine-schedule v-prediction diffusion, losses and deterministic DDIM sampler
class Diffusion:
    def __init__(self, T=1000, s=0.008):
        self.T = T
        x = torch.linspace(0, T, T + 1, dtype=torch.float64)
        ac = torch.cos(((x / T) + s) / (1 + s) * math.pi / 2) ** 2
        ac = ac / ac[0]
        betas = (1 - ac[1:] / ac[:-1]).clamp(max=0.999)
        self.alpha_bar = torch.cumprod(1 - betas, 0).float().to(device)

    def coef(self, t):
        a = self.alpha_bar[t][:, None, None, None]
        return a.sqrt(), (1 - a).sqrt()

diffusion = Diffusion(CFG["timesteps"])

def make_condition(damaged, mask):
    cond = damaged if CFG["use_damaged_pixels"] else damaged * (1 - mask)
    return torch.cat([cond, mask], dim=1)

SOBEL_X = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)

def sobel(img):
    k = SOBEL_X.to(img.device, img.dtype)
    C = img.shape[1]
    gx = F.conv2d(img, k.repeat(C, 1, 1, 1), padding=1, groups=C)
    gy = F.conv2d(img, k.transpose(-1, -2).repeat(C, 1, 1, 1), padding=1, groups=C)
    return gx, gy

def diffusion_loss(net, clean, damaged, mask, t=None, noise=None):
    B = clean.shape[0]
    if t is None:
        t = torch.randint(0, diffusion.T, (B,), device=clean.device)
    if noise is None:
        noise = torch.randn_like(clean)
    sa, s1a = diffusion.coef(t)
    x_t = sa * clean + s1a * noise
    v_target = sa * noise - s1a * clean
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        v_pred = net(torch.cat([x_t, make_condition(damaged, mask)], dim=1), t)
    v_pred = v_pred.float()

    # min-SNR-gamma weighting (v-prediction form) and extra weight on damaged pixels
    snr = (sa / s1a).pow(2).view(B)
    w_t = torch.clamp(snr, max=CFG["min_snr_gamma"]) / (snr + 1)
    w_px = 1.0 + CFG["mask_loss_weight"] * mask
    w_px = w_px / w_px.mean(dim=(1, 2, 3), keepdim=True)
    loss_v = ((v_pred - v_target).pow(2) * w_px).mean(dim=(1, 2, 3))
    loss_v = (w_t * loss_v).mean()

    # auxiliary losses on the predicted clean image, trusted more at low noise levels
    x0 = (sa * x_t - s1a * v_pred).clamp(-1, 1)
    w_aux = (sa.view(B) ** 2)
    m3 = mask.expand_as(clean)
    area = m3.sum(dim=(1, 2, 3)) + 1.0
    l1 = ((x0 - clean).abs() * m3).sum(dim=(1, 2, 3)) / area

    # low-frequency colour loss on the damaged region plus a band around it (fixes colour casts)
    band = F.max_pool2d(mask, 9, stride=1, padding=4).expand_as(clean)
    lf = lambda z: F.avg_pool2d(z, 9, stride=1, padding=4, count_include_pad=False)
    colour = ((lf(x0) - lf(clean)).abs() * band).sum(dim=(1, 2, 3)) / (band.sum(dim=(1, 2, 3)) + 1.0)

    gx_p, gy_p = sobel(x0); gx_c, gy_c = sobel(clean)
    edge = (((gx_p - gx_c).abs() + (gy_p - gy_c).abs()) * m3).sum(dim=(1, 2, 3)) / area

    aux = CFG["lambda_x0_l1"] * l1 + CFG["lambda_colour"] * colour + CFG["lambda_edge"] * edge
    loss = loss_v + (w_aux * aux).mean()
    return loss, {"v": loss_v.item(), "l1": l1.mean().item(), "colour": colour.mean().item(), "edge": edge.mean().item()}

@torch.no_grad()
def ddim_sample(net, damaged, mask, steps=50, seed=0, composite=True):
    # Deterministic DDIM (eta = 0) from fixed-seed noise. Returns images in [0, 1] and the raw full prediction.
    net.eval()
    B = damaged.shape[0]
    gen = torch.Generator(device=damaged.device).manual_seed(seed)
    x = torch.randn(damaged.shape, generator=gen, device=damaged.device)
    cond = make_condition(damaged, mask)
    ts = torch.linspace(diffusion.T - 1, 0, steps, device=damaged.device).round().long()
    for i, t in enumerate(ts):
        tb = torch.full((B,), int(t), device=damaged.device, dtype=torch.long)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            v = net(torch.cat([x, cond], dim=1), tb).float()
        a = diffusion.alpha_bar[t]
        x0 = (a.sqrt() * x - (1 - a).sqrt() * v).clamp(-1, 1)
        eps = (x - a.sqrt() * x0) / (1 - a).sqrt()
        if i < len(ts) - 1:
            a_next = diffusion.alpha_bar[ts[i + 1]]
            x = a_next.sqrt() * x0 + (1 - a_next).sqrt() * eps
        else:
            x = x0
    raw = (x.clamp(-1, 1) + 1) / 2
    known = (damaged + 1) / 2
    out = raw * mask + known * (1 - mask) if composite else raw
    return out, raw

In [6]:
# Cell 6: classical CV colour harmonisation for the reconstructed region
# All functions take float images in [0,1] (H,W,3) and a boolean mask (H,W). `pred` is the model's full-image
# prediction; `known` is the damaged image, whose pixels outside the mask are trusted.

def _ring(mask, width):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * width + 1, 2 * width + 1))
    dil = cv2.dilate(mask.astype(np.uint8), k) > 0
    return dil & ~mask

def composite(pred, known, mask):
    m = mask[..., None].astype(np.float32)
    return pred * m + known * (1 - m)

def feather_blend(pred, known, mask, radius=4):
    # Soft alpha edge so the seam fades over a few pixels instead of a hard step.
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    dil = cv2.dilate(mask.astype(np.uint8), k).astype(np.float32)
    alpha = cv2.GaussianBlur(dil, (0, 0), radius / 2.0)
    alpha = np.maximum(alpha, mask.astype(np.float32))[..., None]
    return pred * alpha + known * (1 - alpha)

def offset_field_correct(pred, known, mask, ring_width=12, sigma=18.0, fallback_k=0.02):
    # Estimate a smooth colour offset (known - pred) on a ring of trusted pixels around each damaged region,
    # spread it into the hole with normalised Gaussian convolution, and add it to the prediction.
    # Only low frequencies change, so text strokes are untouched while colour casts are removed.
    ring = _ring(mask, ring_width)
    if ring.sum() < 20:
        return composite(pred, known, mask)
    r = ring.astype(np.float32)
    diff = (known - pred) * r[..., None]
    num = cv2.GaussianBlur(diff, (0, 0), sigma)
    den = cv2.GaussianBlur(r, (0, 0), sigma)[..., None]
    global_off = diff.sum(axis=(0, 1)) / max(r.sum(), 1.0)
    offset = (num + fallback_k * global_off) / (den + fallback_k)
    corrected = np.clip(pred + offset, 0, 1)
    return composite(corrected, known, mask)

def lab_stats_transfer(pred, known, mask, ring_width=12):
    # Reinhard-style transfer in LAB: match mean/std of the prediction to the known pixels on the surrounding ring.
    ring = _ring(mask, ring_width)
    if ring.sum() < 20:
        return composite(pred, known, mask)
    lp, lk = rgb2lab(pred), rgb2lab(known)
    mu_p, sd_p = lp[ring].mean(0), lp[ring].std(0) + 1e-3
    mu_k, sd_k = lk[ring].mean(0), lk[ring].std(0) + 1e-3
    gain = np.clip(sd_k / sd_p, 0.5, 2.0)
    out = lp.copy()
    out[mask] = (lp[mask] - mu_p) * gain + mu_k
    from skimage.color import lab2rgb
    return composite(np.clip(lab2rgb(out), 0, 1), known, mask)

def poisson_blend(pred, known, mask, pad=16):
    # Gradient-domain (Poisson) blending with OpenCV seamlessClone: keeps the predicted gradients (text edges)
    # and solves for colours that agree with the known pixels on the boundary.
    if mask.sum() < 10:
        return composite(pred, known, mask)
    to8 = lambda a: (np.clip(a, 0, 1) * 255 + 0.5).astype(np.uint8)
    src = cv2.copyMakeBorder(to8(pred), pad, pad, pad, pad, cv2.BORDER_REFLECT)
    dst = cv2.copyMakeBorder(to8(known), pad, pad, pad, pad, cv2.BORDER_REFLECT)
    m = cv2.copyMakeBorder(mask.astype(np.uint8) * 255, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0)
    m = cv2.dilate(m, np.ones((3, 3), np.uint8))
    x, y, w, h = cv2.boundingRect(m)
    center = (x + w // 2, y + h // 2)
    try:
        out = cv2.seamlessClone(src, dst, m, center, cv2.NORMAL_CLONE)
    except cv2.error:
        return composite(pred, known, mask)
    out = out[pad:-pad, pad:-pad].astype(np.float32) / 255.0
    return composite(out, known, mask)

def offset_then_feather(pred, known, mask):
    ring = _ring(mask, 12)
    if ring.sum() < 20:
        return feather_blend(pred, known, mask)
    r = ring.astype(np.float32)
    diff = (known - pred) * r[..., None]
    num = cv2.GaussianBlur(diff, (0, 0), 18.0)
    den = cv2.GaussianBlur(r, (0, 0), 18.0)[..., None]
    g = diff.sum(axis=(0, 1)) / max(r.sum(), 1.0)
    corrected = np.clip(pred + (num + 0.02 * g) / (den + 0.02), 0, 1)
    return feather_blend(corrected, known, mask)

def offset_then_poisson(pred, known, mask):
    ring = _ring(mask, 12)
    if ring.sum() < 20:
        return poisson_blend(pred, known, mask)
    r = ring.astype(np.float32)
    diff = (known - pred) * r[..., None]
    num = cv2.GaussianBlur(diff, (0, 0), 18.0)
    den = cv2.GaussianBlur(r, (0, 0), 18.0)[..., None]
    g = diff.sum(axis=(0, 1)) / max(r.sum(), 1.0)
    corrected = np.clip(pred + (num + 0.02 * g) / (den + 0.02), 0, 1)
    return poisson_blend(corrected, known, mask)

HARMONIZERS = {
    "raw_composite": composite,
    "offset_field": offset_field_correct,
    "lab_stats": lab_stats_transfer,
    "poisson": poisson_blend,
    "offset+poisson": offset_then_poisson,
}
print("Harmonizers:", list(HARMONIZERS))

Harmonizers: ['raw_composite', 'offset_field', 'lab_stats', 'poisson', 'offset+poisson']


In [7]:
# Cell 7: metrics (full / masked PSNR, SSIM, pixel accuracy, colour error dE2000) and baselines
def psnr(a, b):
    mse = float(np.mean((a - b) ** 2))
    return 100.0 if mse == 0 else 10 * math.log10(1.0 / mse)

def image_metrics(pred, clean, mask):
    m = mask.astype(bool)
    out = {"psnr": psnr(pred, clean),
           "ssim": structural_similarity(pred, clean, channel_axis=2, data_range=1.0),
           "pix_acc": float(np.mean(np.abs(pred - clean) < 0.15) * 100)}
    if m.sum() > 0:
        out["masked_psnr"] = psnr(pred[m], clean[m])
        out["masked_pix_acc"] = float(np.mean(np.abs(pred[m] - clean[m]) < 0.15) * 100)
        ys, xs = np.nonzero(m)
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        if (y1 - y0) >= 7 and (x1 - x0) >= 7:
            out["masked_bbox_ssim"] = structural_similarity(pred[y0:y1, x0:x1], clean[y0:y1, x0:x1],
                                                            channel_axis=2, data_range=1.0)
        out["masked_deltaE"] = float(np.mean(deltaE_ciede2000(rgb2lab(pred)[m], rgb2lab(clean)[m])))
    return out

def telea_baseline(known, mask):
    img8 = (np.clip(known, 0, 1) * 255 + 0.5).astype(np.uint8)
    out = cv2.inpaint(img8, mask.astype(np.uint8) * 255, 3, cv2.INPAINT_TELEA)
    return out.astype(np.float32) / 255.0

def summarise(rows):
    keys = ["masked_psnr", "masked_bbox_ssim", "masked_pix_acc", "masked_deltaE", "psnr", "ssim", "pix_acc"]
    res = {}
    for k in keys:
        vals = [r[k] for r in rows if k in r and np.isfinite(r[k])]
        res[k] = (float(np.mean(vals)), float(np.std(vals))) if vals else (float("nan"), float("nan"))
    return res

to_np = lambda t: t.permute(1, 2, 0).cpu().float().numpy()

In [8]:
# Cell 8: OCR tools
#   EasyOCR  -> text-line detection + the reader used for EVALUATION (independent of training)
#   TrOCR    -> frozen, differentiable reader used for the TEXT LOSS and for RE-RANKING samples
import sys, subprocess
from torchvision.ops import roi_align

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import easyocr
except ImportError:
    # --no-deps so pip does not replace Kaggle's torch / opencv builds
    _pip("--no-deps", "easyocr"); _pip("python-bidi", "pyclipper", "shapely", "ninja")
    import easyocr
try:
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
except ImportError:
    _pip("transformers"); from transformers import TrOCRProcessor, VisionEncoderDecoderModel

reader = easyocr.Reader(["en"], gpu=(device.type == "cuda"), verbose=False)

trocr_proc = TrOCRProcessor.from_pretrained(CFG["trocr_name"])
trocr = VisionEncoderDecoderModel.from_pretrained(CFG["trocr_name"]).to(device).eval()
for p in trocr.parameters():
    p.requires_grad_(False)
tok = trocr_proc.tokenizer
def _first(*vals):
    return next((v for v in vals if v is not None), None)
_cfg, _gen, _dec = trocr.config, getattr(trocr, "generation_config", None), trocr.config.decoder
EOS_ID = _first(getattr(_dec, "eos_token_id", None), getattr(_gen, "eos_token_id", None), tok.eos_token_id)
PAD_ID = _first(getattr(_cfg, "pad_token_id", None), getattr(_gen, "pad_token_id", None),
                getattr(_dec, "pad_token_id", None), tok.pad_token_id)
START_ID = _first(getattr(_cfg, "decoder_start_token_id", None), getattr(_gen, "decoder_start_token_id", None),
                  getattr(_dec, "decoder_start_token_id", None), EOS_ID)
print(f"TrOCR token ids: start {START_ID} | eos {EOS_ID} | pad {PAD_ID}")
_ip = trocr_proc.image_processor
def _size_hw(size, default=384):
    # works for int, plain dict and transformers' SizeDict
    if isinstance(size, (int, float)):
        return int(size), int(size)
    get = (lambda k: size.get(k)) if isinstance(size, dict) else (lambda k: getattr(size, k, None))
    h, w = get("height"), get("width")
    if h is None or w is None:
        e = get("shortest_edge") or get("longest_edge") or default
        h, w = (h or e), (w or e)
    return int(h), int(w)
TROCR_HW = _size_hw(_ip.size)
TROCR_MEAN = torch.tensor(_ip.image_mean, device=device).view(1, 3, 1, 1)
TROCR_STD = torch.tensor(_ip.image_std, device=device).view(1, 3, 1, 1)
print(f"EasyOCR ready | TrOCR '{CFG['trocr_name']}' ready: input {TROCR_HW}, "
      f"{sum(p.numel() for p in trocr.parameters())/1e6:.1f}M params (frozen)")

# ---------------- text helpers ----------------
def norm_text(s):
    # case-insensitive, letters and digits only: "Paracetamol 500 mg." -> "paracetamol500mg"
    return "".join(ch for ch in str(s).lower() if ch.isalnum())

def levenshtein(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def cer(pred, ref):
    p, r = norm_text(pred), norm_text(ref)
    return min(levenshtein(p, r) / max(len(r), 1), 1.0)   # clipped at 100 %

def encode_label(text):
    ids = tok(text, add_special_tokens=False).input_ids[:CFG["text_max_len"] - 1]
    return ids + [EOS_ID]

# ---------------- TrOCR (differentiable crops) ----------------
def make_rois(batch_idx, boxes_xyxy, pad=2.0):
    b = boxes_xyxy.float().clone()
    b[:, :2] -= pad; b[:, 2:] += pad
    return torch.cat([batch_idx.float()[:, None], b], dim=1)

def trocr_crops(img_m11, rois):
    # img in [-1, 1] (B,3,H,W) -> TrOCR pixel_values (K,3,384,384); differentiable w.r.t. img
    crops = roi_align(img_m11.float(), rois.to(img_m11.device), output_size=TROCR_HW,
                      spatial_scale=1.0, sampling_ratio=2, aligned=True)
    return (((crops + 1) / 2) - TROCR_MEAN) / TROCR_STD

def trocr_token_nll(pixel_values, labels):
    # per-crop mean token cross-entropy (teacher forcing); labels (K,L) padded with -100.
    # decoder inputs are built here (start token + labels shifted right), so no config fields are needed
    dec_in = torch.cat([torch.full_like(labels[:, :1], START_ID), labels[:, :-1]], dim=1)
    dec_in = dec_in.masked_fill(dec_in == -100, PAD_ID)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        logits = trocr(pixel_values=pixel_values, decoder_input_ids=dec_in).logits
    ce = F.cross_entropy(logits.float().transpose(1, 2), labels, ignore_index=-100, reduction="none")
    valid = (labels != -100).float()
    return (ce * valid).sum(1) / valid.sum(1).clamp(min=1.0)

@torch.no_grad()
def trocr_read(pixel_values, bs=64):
    # greedy decode -> (texts, mean token log-prob per crop)
    texts, conf = [], []
    for s in range(0, pixel_values.shape[0], bs):
        pv = pixel_values[s:s + bs]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            out = trocr.generate(pv, max_new_tokens=CFG["text_max_len"], num_beams=1, do_sample=False,
                                 decoder_start_token_id=START_ID, eos_token_id=EOS_ID, pad_token_id=PAD_ID,
                                 output_scores=True, return_dict_in_generate=True)
        lp = trocr.compute_transition_scores(out.sequences, out.scores, normalize_logits=True).float()
        gen = out.sequences[:, -lp.shape[1]:]
        is_eos = (gen == EOS_ID).int()
        valid = ((is_eos.cumsum(1) - is_eos) == 0) & torch.isfinite(lp)
        conf += ((lp.nan_to_num(0) * valid).sum(1) / valid.sum(1).clamp(min=1)).tolist()
        texts += tok.batch_decode(out.sequences, skip_special_tokens=True)
    return texts, conf

# ---------------- EasyOCR (evaluation reader) ----------------
def to_u8(a):
    return (np.clip(a, 0, 1) * 255 + 0.5).astype(np.uint8)

def easy_detect_lines(img01):
    # axis-aligned text-line boxes [x0, y0, x1, y1] on a [0,1] RGB image
    h, w = img01.shape[:2]
    hor, _ = reader.detect(to_u8(img01), mag_ratio=CFG["ocr_mag_ratio"])
    out = []
    for x0, x1, y0, y1 in (hor[0] if hor else []):
        x0, y0, x1, y1 = max(int(x0), 0), max(int(y0), 0), min(int(x1), w), min(int(y1), h)
        if x1 - x0 >= 6 and y1 - y0 >= 5:
            out.append([x0, y0, x1, y1])
    return out

def easy_read_boxes(img01, boxes):
    # read every box of one image in a single call; results matched back to the boxes by coordinates
    if not boxes:
        return []
    gray = cv2.cvtColor(to_u8(img01), cv2.COLOR_RGB2GRAY)
    hl = [[int(b[0]), int(b[2]), int(b[1]), int(b[3])] for b in boxes]
    res = reader.recognize(gray, horizontal_list=hl, free_list=[], detail=1, paragraph=False, batch_size=16)
    by_box = {}
    for pts, text, _conf in res:
        (x0, y0), (x1, y1) = pts[0], pts[2]
        by_box[(int(x0), int(y0), int(x1), int(y1))] = text
    return [by_box.get((h_[0], h_[2], h_[1], h_[3]), "") for h_ in hl]

def damaged_entries(entries, mask, min_frac=None):
    # entries whose box is at least `min_frac` covered by the damage mask
    min_frac = CFG["box_min_damage"] if min_frac is None else min_frac
    out = []
    for e in entries:
        x0, y0, x1, y1 = e["box"]
        if x1 > x0 and y1 > y0 and mask[y0:y1, x0:x1].mean() >= min_frac:
            out.append(e)
    return out

preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 1.36MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TrOCR token ids: start 2 | eos 2 | pad 1
EasyOCR ready | TrOCR 'microsoft/trocr-small-printed' ready: input (384, 384), 61.6M params (frozen)


In [9]:
# Cell 9: OCR pre-pass on every CLEAN label (run once, cached to ocr_cache_v3.json)
#   1. EasyOCR detects and reads the text lines of the clean 416x416 label  -> evaluation boxes
#   2. TrOCR reads the same lines; lines where both engines agree (CER <= consensus_cer) become TRAINING labels
#      (label = TrOCR's own reading, so the text loss is ~0 on the clean image by construction)
def find_input_file(name):
    hits = sorted(INPUT_DIR.rglob(name)) if INPUT_DIR.exists() else []
    return hits[0] if hits else None

all_clean = sorted({s["clean"] for s in train_samples + val_samples + test_samples})
cache_src = find_input_file("ocr_cache_v3.json") if CFG["ocr_cache"] == "auto" else (Path(CFG["ocr_cache"]) if CFG["ocr_cache"] else None)
ocr_cache = {}
if cache_src is not None and Path(cache_src).exists():
    with open(cache_src) as f:
        ocr_cache = json.load(f)
    print(f"Loaded OCR cache from {cache_src}: {len(ocr_cache)} clean labels")

todo = [f for f in all_clean if f not in ocr_cache]
if todo:
    print(f"OCR pre-pass on {len(todo)} clean labels (budget {CFG['ocr_cache_max_hours']:.2f} h)...")
    t0 = time.time()
    for n, f in enumerate(todo):
        if time.time() - t0 > CFG["ocr_cache_max_hours"] * 3600:
            print(f"  cache time budget reached after {n} labels; the rest get no text boxes"); break
        img = read_rgb(DATA_ROOT / "clean" / f, CFG["image_size"])
        res = reader.readtext(to_u8(img), mag_ratio=CFG["ocr_mag_ratio"], detail=1, paragraph=False)
        entries = []
        for pts, text, conf in res:
            xs, ys = [p[0] for p in pts], [p[1] for p in pts]
            x0, y0 = max(int(min(xs)), 0), max(int(min(ys)), 0)
            x1, y1 = min(int(max(xs)), CFG["image_size"]), min(int(max(ys)), CFG["image_size"])
            if conf >= CFG["ocr_min_conf"] and len(norm_text(text)) >= 2 and x1 - x0 >= 6 and y1 - y0 >= 5:
                entries.append({"box": [x0, y0, x1, y1], "easy": text, "conf": round(float(conf), 3)})
        # --- TrOCR consensus on the same lines ---
        if entries:
            ct = torch.from_numpy(img).permute(2, 0, 1)[None].to(device) * 2 - 1
            boxes = torch.tensor([e["box"] for e in entries], device=device)
            tr_text, tr_conf = trocr_read(trocr_crops(ct, make_rois(torch.zeros(len(entries), device=device), boxes)))
            for e, tt, tc in zip(entries, tr_text, tr_conf):
                agree = cer(tt, e["easy"]) <= CFG["consensus_cer"] and len(norm_text(tt)) >= 2
                e.update(trocr=tt, trocr_lp=round(float(tc), 3), keep=bool(agree),
                         ids=encode_label(tt) if agree else [])
        ocr_cache[f] = entries
        if (n + 1) % 500 == 0:
            el = time.time() - t0
            print(f"  {n+1}/{len(todo)} labels | {el/60:.1f} min | ETA {(len(todo)-n-1)*el/(n+1)/60:.0f} min")
            with open(OUT_DIR / "ocr_cache_v3.json", "w") as fh:
                json.dump(ocr_cache, fh)
    with open(OUT_DIR / "ocr_cache_v3.json", "w") as fh:
        json.dump(ocr_cache, fh)
    print(f"OCR pre-pass done in {(time.time()-t0)/60:.1f} min -> {OUT_DIR/'ocr_cache_v3.json'} "
          f"(attach it as a dataset next time to skip this step)")

n_lines = sum(len(v) for v in ocr_cache.values())
n_keep = sum(e.get("keep", False) for v in ocr_cache.values() for e in v)
print(f"Text lines: {n_lines} ({n_lines/max(len(ocr_cache),1):.1f} per label) | EasyOCR/TrOCR agree on "
      f"{n_keep} ({100*n_keep/max(n_lines,1):.1f} %) -> used as text-loss labels")
if n_lines and n_keep / n_lines < 0.2:
    print("  WARNING: low agreement. Look at the examples below; consider CFG['trocr_name']='microsoft/trocr-base-printed' "
          "or a larger consensus_cer.")
ex = [e for v in list(ocr_cache.values())[:50] for e in v][:8]
for e in ex:
    print(f"  box {e['box']} | EasyOCR '{e['easy']}' | TrOCR '{e.get('trocr', '')}' | keep {e.get('keep')}")

# how much text does the damage actually hit? (test split)
hit = [len(damaged_entries(ocr_cache.get(s["clean"], []), read_mask(DATA_ROOT / "masks" / s["mask"], CFG["image_size"])))
       for s in test_samples[::max(1, len(test_samples) // 300)]]
print(f"Test labels with >=1 damaged text line: {100*np.mean([h > 0 for h in hit]):.1f} % | "
      f"mean damaged lines per label: {np.mean(hit):.2f}")


Loaded OCR cache from /kaggle/input/datasets/ayushsinghh1086/ocr-cache-v3/ocr_cache_v3.json: 6957 clean labels
Text lines: 6294 (0.9 per label) | EasyOCR/TrOCR agree on 3306 (52.5 %) -> used as text-loss labels
  box [0, 243, 71, 258] | EasyOCR '0 tablets' | TrOCR 'O TABLETS' | keep True
  box [88, 217, 178, 243] | EasyOCR '85*912' | TrOCR '216.58' | keep False
  box [260, 245, 317, 297] | EasyOCR '12' | TrOCR ':' | keep False
  box [188, 121, 278, 177] | EasyOCR '883' | TrOCR '***' | keep False
  box [212, 176, 277, 245] | EasyOCR ''28' | TrOCR '*' | keep False
  box [209, 153, 262, 204] | EasyOCR '33' | TrOCR '%' | keep False
  box [10, 171, 112, 246] | EasyOCR 'pM' | TrOCR '***' | keep False
  box [25, 107, 209, 144] | EasyOCR 'Dagagilozin Tablets 5 mg' | TrOCR 'DAPAGLITILOZN TABLETS 5 MG' | keep True
Test labels with >=1 damaged text line: 17.5 % | mean damaged lines per label: 0.31


In [10]:
# Cell 10: load the v2 weights, plus the shared sampling / re-ranking / text-accuracy functions

# fast LAB harmoniser (the one selected on validation in v2)
from skimage.color import lab2rgb
def lab_stats_transfer(pred, known, mask, ring_width=12):
    ring = _ring(mask, ring_width)
    if ring.sum() < 20 or mask.sum() == 0:
        return composite(pred, known, mask)
    lp_r, lk_r = rgb2lab(pred[ring][None])[0], rgb2lab(known[ring][None])[0]
    lp_m = rgb2lab(pred[mask][None])[0]
    mu_p, sd_p = lp_r.mean(0), lp_r.std(0) + 1e-3
    mu_k, sd_k = lk_r.mean(0), lk_r.std(0) + 1e-3
    new = (lp_m - mu_p) * np.clip(sd_k / sd_p, 0.5, 2.0) + mu_k
    out = known.copy()
    out[mask] = np.clip(lab2rgb(new[None])[0], 0, 1)
    return out
HARMONIZERS["lab_stats"] = lab_stats_transfer
best_h = CFG["harmonizer"]
harmonise = HARMONIZERS[best_h]

# ---------- weights ----------
strip = lambda sd: {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in sd.items()}
ck = torch.load(RESUME_PATH, map_location=device, weights_only=False)
old_cfg = ck.get("cfg", {})
for k in ["base_channels", "channel_mults", "num_res_blocks", "attn_levels", "image_size", "use_damaged_pixels"]:
    if k in old_cfg and tuple(np.atleast_1d(old_cfg[k])) != tuple(np.atleast_1d(CFG[k])):
        raise ValueError(f"Checkpoint was trained with {k}={old_cfg[k]} but CFG has {CFG[k]}")
base_sd = strip(ck["ema"])                      # v2 EMA weights = the baseline and the fine-tuning start point
prev_state = ck.get("state", {})
del ck
model.load_state_dict(base_sd)
baseline = copy.deepcopy(model).eval()
for p in baseline.parameters():
    p.requires_grad_(False)
base_net = (nn.DataParallel(baseline) if NUM_GPUS > 1 else baseline).eval()
print(f"Loaded v2 EMA weights from {RESUME_PATH} (epoch {prev_state.get('epoch', '?')}, "
      f"val masked PSNR {prev_state.get('best_val_masked_psnr', float('nan')):.2f} dB)")

# ---------- index of images whose damage hits text ----------
def text_hit_indices(samples, n, need_keep=False):
    ok = []
    for i, s in enumerate(samples):
        ents = [e for e in ocr_cache.get(s["clean"], []) if e.get("keep") or not need_keep]
        if ents and damaged_entries(ents, read_mask(DATA_ROOT / "masks" / s["mask"], CFG["image_size"])):
            ok.append(i)
    if len(ok) > n:
        ok = [ok[j] for j in np.linspace(0, len(ok) - 1, n).round().astype(int)]
    return ok

class IndexedSubset(Dataset):
    # like Subset, but also returns the index into the parent dataset (needed to look up the OCR cache)
    def __init__(self, dataset, indices):
        self.dataset, self.indices = dataset, list(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        return (*self.dataset[self.indices[i]], self.indices[i])

# ---------- re-ranking ----------
@torch.no_grad()
def sample_candidates(netw, d, m, n, steps):
    return [ddim_sample(netw, d, m, steps=steps, seed=s)[1] for s in range(n)]   # list of raw (B,3,H,W) in [0,1]

@torch.no_grad()
def rerank(cands, d, m, det_boxes):
    # pick, per image, the candidate whose damaged text lines TrOCR reads most confidently.
    # det_boxes: text lines detected on the DAMAGED input (no ground truth involved)
    B, N = d.shape[0], len(cands)
    known = (d + 1) / 2
    pick = torch.zeros(B, dtype=torch.long)
    for i in range(B):
        mm = m[i, 0].cpu().numpy() > 0.5
        boxes = [b["box"] for b in damaged_entries([{"box": b} for b in det_boxes[i]], mm)]
        if not boxes or N == 1:
            continue
        bt = torch.tensor(boxes, device=d.device)
        stack = torch.stack([c[i] * m[i] + known[i] * (1 - m[i]) for c in cands]) * 2 - 1   # (N,3,H,W)
        rois = torch.cat([make_rois(torch.full((len(boxes),), j, device=d.device), bt) for j in range(N)])
        _, conf = trocr_read(trocr_crops(stack, rois))
        pick[i] = int(np.argmax(np.array(conf).reshape(N, len(boxes)).mean(1)))
    return torch.stack([cands[int(pick[i])][i] for i in range(B)]), pick

# ---------- text accuracy on the damaged lines ----------
_ref_memo = {}
def text_scores(img01, clean01, clean_file, mask):
    # per damaged GT line: CER and exact-word match of EasyOCR(img) vs EasyOCR(clean), same crop
    ents = damaged_entries(ocr_cache.get(clean_file, []), mask)
    if not ents:
        return []
    boxes = [e["box"] for e in ents]
    key = clean_file
    if key not in _ref_memo:
        _ref_memo[key] = {}
    missing = [b for b in boxes if tuple(b) not in _ref_memo[key]]
    for b, t in zip(missing, easy_read_boxes(clean01, missing)):
        _ref_memo[key][tuple(b)] = t
    refs = [_ref_memo[key][tuple(b)] for b in boxes]
    preds = easy_read_boxes(img01, boxes)
    return [{"cer": cer(p, r), "exact": float(norm_text(p) == norm_text(r)), "ref": r, "pred": p, "box": b}
            for p, r, b in zip(preds, refs, boxes) if len(norm_text(r)) >= 2]
print("Evaluation tools ready | harmoniser:", best_h)


Loaded v2 EMA weights from /kaggle/input/datasets/ayushsinghh1086/416pathbestema/best_ema.pth (epoch ?, val masked PSNR nan dB)
Evaluation tools ready | harmoniser: lab_stats


In [11]:
# Cell 11: OCR-aware fine-tuning = v2 losses + lambda_text * TrOCR cross-entropy on the damaged text lines
class OCRTrainDataset(PharmacyLabelDataset):
    # same damage-centred crops as v2, plus up to K damaged text lines (crop coordinates) and their token labels
    def __init__(self, *a, cache=None, **k):
        super().__init__(*a, **k)
        self.cache, self.K, self.L = cache, CFG["text_k_per_img"], CFG["text_max_len"]

    def __getitem__(self, i):
        it = self.samples[i]
        clean = read_rgb(self.root / "clean" / it["clean"], self.image_size)
        damaged = read_rgb(self.root / "damaged" / it["damaged"], self.image_size)
        mask = read_mask(self.root / "masks" / it["mask"], self.image_size)
        y0, x0 = self._crop_window(mask) if (self.train and self.crop_size) else (0, 0)
        c = self.crop_size if (self.train and self.crop_size) else self.image_size
        clean, damaged, mask = clean[y0:y0+c, x0:x0+c], damaged[y0:y0+c, x0:x0+c], mask[y0:y0+c, x0:x0+c]
        boxes = np.full((self.K, 4), -1.0, np.float32)
        labels = np.full((self.K, self.L), -100, np.int64)
        cands = []
        for e in self.cache.get(it["clean"], []):
            if not e.get("keep"):
                continue
            bx0, by0, bx1, by1 = e["box"]
            cx0, cy0, cx1, cy1 = max(bx0 - x0, 0), max(by0 - y0, 0), min(bx1 - x0, c), min(by1 - y0, c)
            if cx1 - cx0 < 6 or cy1 - cy0 < 5 or (cx1 - cx0) * (cy1 - cy0) < 0.9 * (bx1 - bx0) * (by1 - by0):
                continue                                   # line (almost) fully inside the crop only
            if mask[cy0:cy1, cx0:cx1].mean() < CFG["box_min_damage"]:
                continue                                   # only lines the damage actually touches
            cands.append(([cx0, cy0, cx1, cy1], e["ids"]))
        random.shuffle(cands)
        for j, (b, ids) in enumerate(cands[:self.K]):
            boxes[j] = b; labels[j, :len(ids)] = ids[:self.L]
        to_t = lambda a: torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1) * 2.0 - 1.0
        return (to_t(clean), to_t(damaged), torch.from_numpy(np.ascontiguousarray(mask))[None],
                torch.from_numpy(boxes), torch.from_numpy(labels))

def diffusion_loss_ocr(net, clean, damaged, mask, boxes, labels):
    B = clean.shape[0]
    t = torch.randint(0, diffusion.T, (B,), device=clean.device)
    noise = torch.randn_like(clean)
    sa, s1a = diffusion.coef(t)
    x_t = sa * clean + s1a * noise
    v_target = sa * noise - s1a * clean
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        v_pred = net(torch.cat([x_t, make_condition(damaged, mask)], dim=1), t)
    v_pred = v_pred.float()
    # ---- v2 losses (unchanged) ----
    snr = (sa / s1a).pow(2).view(B)
    w_t = torch.clamp(snr, max=CFG["min_snr_gamma"]) / (snr + 1)
    w_px = 1.0 + CFG["mask_loss_weight"] * mask
    w_px = w_px / w_px.mean(dim=(1, 2, 3), keepdim=True)
    loss_v = (w_t * ((v_pred - v_target).pow(2) * w_px).mean(dim=(1, 2, 3))).mean()
    x0 = (sa * x_t - s1a * v_pred).clamp(-1, 1)
    w_aux = sa.view(B) ** 2
    m3 = mask.expand_as(clean)
    area = m3.sum(dim=(1, 2, 3)) + 1.0
    l1 = ((x0 - clean).abs() * m3).sum(dim=(1, 2, 3)) / area
    band = F.max_pool2d(mask, 9, stride=1, padding=4).expand_as(clean)
    lf = lambda z: F.avg_pool2d(z, 9, stride=1, padding=4, count_include_pad=False)
    colour = ((lf(x0) - lf(clean)).abs() * band).sum(dim=(1, 2, 3)) / (band.sum(dim=(1, 2, 3)) + 1.0)
    gx_p, gy_p = sobel(x0); gx_c, gy_c = sobel(clean)
    edge = (((gx_p - gx_c).abs() + (gy_p - gy_c).abs()) * m3).sum(dim=(1, 2, 3)) / area
    aux = CFG["lambda_x0_l1"] * l1 + CFG["lambda_colour"] * colour + CFG["lambda_edge"] * edge
    loss = loss_v + (w_aux * aux).mean()
    # ---- OCR text loss on the damaged lines of the composited x0 (low-noise samples only) ----
    text = torch.zeros((), device=clean.device)
    sel = (boxes[..., 0] >= 0) & (w_aux >= CFG["text_min_abar"])[:, None]
    idx = sel.nonzero()
    if idx.shape[0] > CFG["text_max_boxes"]:
        idx = idx[torch.randperm(idx.shape[0], device=idx.device)[:CFG["text_max_boxes"]]]
    if idx.shape[0] > 0:
        x0c = x0 * mask + damaged * (1 - mask)          # gradients only flow into the damaged pixels
        bi, ki = idx[:, 0], idx[:, 1]
        nll = trocr_token_nll(trocr_crops(x0c, make_rois(bi, boxes[bi, ki])), labels[bi, ki])
        text = (w_aux[bi] * nll).sum() / w_aux[bi].sum().clamp(min=1e-6)
        loss = loss + CFG["lambda_text"] * text
    return loss, {"v": loss_v.item(), "l1": l1.mean().item(), "edge": edge.mean().item(),
                  "text": text.item(), "n_lines": int(idx.shape[0])}

# ---------- validation: masked PSNR + text accuracy on val images whose damage hits text ----------
val_ocr_idx = text_hit_indices(val_samples, CFG["ocr_val_images"])
val_ocr_loader = DataLoader(IndexedSubset(val_ds, val_ocr_idx), batch_size=CFG["eval_batch_size"], num_workers=CFG["num_workers"])
print(f"Validation for fine-tuning: {len(val_ocr_idx)} val labels with damaged text")

@torch.no_grad()
def validate_text(netw, steps):
    psnrs, cers, exact = [], [], []
    for c, d, m, idx in val_ocr_loader:
        _, raw = ddim_sample(netw, d.to(device), m.to(device), steps=steps, seed=0)
        for i in range(c.shape[0]):
            mm = m[i, 0].numpy() > 0.5
            clean_i, known_i = to_np((c[i] + 1) / 2), to_np((d[i] + 1) / 2)
            out = composite(to_np(raw[i]), known_i, mm)
            psnrs.append(psnr(out[mm], clean_i[mm]))
            for r in text_scores(out, clean_i, val_samples[int(idx[i])]["clean"], mm):
                cers.append(r["cer"]); exact.append(r["exact"])
    return float(np.mean(psnrs)), float(np.mean(cers)) if cers else float("nan"), float(np.mean(exact)) if exact else float("nan")

base_val = validate_text(base_net, CFG["val_psnr_steps"])
print(f"Baseline (v2) on this val subset: masked PSNR {base_val[0]:.2f} dB | line CER {100*base_val[1]:.2f} % | "
      f"word acc {100*base_val[2]:.2f} %")

FT_PATH = OUT_DIR / "best_ema_ocr.pth"
ft_budget_s = max(0.0, min(CFG["FT_HOURS"], hours_left() - CFG["eval_reserve_hours"])) * 3600
ft_history = []
if ft_budget_s < (600 if not QUICK_TEST else 1):
    print(f"Not enough time left to fine-tune ({hours_left():.2f} h left); skipping to evaluation with the baseline only")
else:
    train_files = {s_["clean"] for s_ in train_samples}
    ft_cache = {f: [{"box": tuple(e["box"]), "ids": tuple(e["ids"]), "keep": True} for e in ocr_cache.get(f, []) if e.get("keep")]
                for f in train_files}
    ft_cache = {f: v for f, v in ft_cache.items() if v}
    ft_ds = OCRTrainDataset(DATA_ROOT, train_samples, CFG["image_size"], CFG["crop_size"], train=True, cache=ft_cache)
    gc.collect()
    make_loader = lambda bs: DataLoader(ft_ds, batch_size=bs, shuffle=True, drop_last=True, num_workers=CFG["num_workers"],
                                        pin_memory=(device.type == "cuda"), persistent_workers=False)
    model.load_state_dict(base_sd); model.train()
    ema = copy.deepcopy(model).eval()
    for p in ema.parameters():
        p.requires_grad_(False)
    ema_net = (nn.DataParallel(ema) if NUM_GPUS > 1 else ema)
    net = nn.DataParallel(model) if NUM_GPUS > 1 else model
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["ft_lr"], weight_decay=CFG["weight_decay"], betas=(0.9, 0.99))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    # batch-size probe with the full loss (TrOCR adds memory on GPU 0)
    bs = CFG["batch_size"]
    while True:
        try:
            batch = next(iter(make_loader(bs)))
            batch = [x.to(device) for x in batch]
            loss, _ = diffusion_loss_ocr(net, *batch)
            loss.backward(); optimizer.zero_grad(set_to_none=True)
            print(f"Fine-tune batch size {bs} fits")
            break
        except torch.OutOfMemoryError:
            optimizer.zero_grad(set_to_none=True); torch.cuda.empty_cache()
            if bs <= 2:
                raise
            bs //= 2; print(f"OOM -> batch size {bs}")
    torch.cuda.empty_cache() if device.type == "cuda" else None
    loader = make_loader(bs)

    @torch.no_grad()
    def ema_step():
        dcy = CFG["ema_decay"]
        for pe, pm in zip(ema.parameters(), model.parameters()):
            pe.mul_(dcy).add_(pm.detach(), alpha=1 - dcy)
        for be, bm in zip(ema.buffers(), model.buffers()):
            be.copy_(bm)

    def lr_ft(step, progress):
        if step < CFG["ft_warmup_steps"]:
            return CFG["ft_lr"] * (step + 1) / CFG["ft_warmup_steps"]
        return CFG["min_lr"] + 0.5 * (CFG["ft_lr"] - CFG["min_lr"]) * (1 + math.cos(math.pi * min(progress, 1.0)))

    best = {"cer": base_val[1], "psnr": base_val[0], "saved": False}
    def run_val(step, el):
        vp, vc, ve = validate_text(ema_net, CFG["val_psnr_steps"])
        ok = vp >= base_val[0] - CFG["max_val_psnr_drop"]
        better = ok and vc < best["cer"]
        ft_history.append({"step": step, "hours": el / 3600, "val_masked_psnr": vp, "val_cer": vc, "val_word_acc": ve})
        print(f"  [val] step {step} | masked PSNR {vp:.2f} dB | line CER {100*vc:.2f} % | word acc {100*ve:.2f} %"
              + ("  -> new best, saved" if better else ("" if ok else "  (PSNR dropped too much, not eligible)")))
        if better:
            best.update(cer=vc, psnr=vp, saved=True)
            torch.save({"ema": ema.state_dict(), "cfg": CFG, "step": step, "val": ft_history[-1]}, FT_PATH)
        with open(OUT_DIR / "ft_history.json", "w") as fh:
            json.dump(ft_history, fh, indent=1)
        torch.save({"ema": ema.state_dict(), "cfg": CFG, "step": step}, OUT_DIR / "last_ema_ocr.pth")
        gc.collect()

    print(f"RAM before fine-tuning: {ram_percent():.0f} %")
    print(f"OCR fine-tuning for {ft_budget_s/3600:.2f} h | batch {bs} | lambda_text {CFG['lambda_text']} | lr {CFG['ft_lr']:.1e}")
    t_start, last_val, step, stop = time.time(), time.time(), 0, False
    run_log = {"text": [], "n": []}
    while not stop:
        for batch in loader:
            el = time.time() - t_start
            if el >= ft_budget_s:
                stop = True; break
            if step % 50 == 0 and ram_percent() > CFG["ram_stop_percent"]:
                print(f"  RAM at {ram_percent():.0f} % -> stopping fine-tuning early to protect the evaluation")
                stop = True; break
            batch = [x.to(device, non_blocking=True) for x in batch]
            for g_ in optimizer.param_groups:
                g_["lr"] = lr_ft(step, el / ft_budget_s)
            loss, parts = diffusion_loss_ocr(net, *batch)
            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True); continue
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(optimizer); scaler.update(); ema_step(); step += 1
            if parts["n_lines"]:
                run_log["text"].append(parts["text"]); run_log["n"].append(parts["n_lines"])
            if step % (200 if not QUICK_TEST else 1) == 0:
                tl = np.mean(run_log["text"][-200:]) if run_log["text"] else float("nan")
                print(f"  step {step:5d} | loss {loss.item():.4f} | v {parts['v']:.4f} | l1 {parts['l1']:.4f} | "
                      f"text {tl:.3f} ({np.mean(run_log['n'][-200:]) if run_log['n'] else 0:.1f} lines/batch) | "
                      f"lr {optimizer.param_groups[0]['lr']:.2e} | {el/3600:.2f} h | RAM {ram_percent():.0f} %")
            if time.time() - last_val > CFG["ft_val_every_min"] * 60:
                run_val(step, el); last_val = time.time(); model.train()
    run_val(step, time.time() - t_start)
    torch.save({"ema": ema.state_dict(), "cfg": CFG, "step": step}, OUT_DIR / "last_ema_ocr.pth")   # always kept, even if not "best"
    print(f"Fine-tuning finished: {step} steps | best val line CER {100*best['cer']:.2f} % "
          f"(baseline {100*base_val[1]:.2f} %) | saved: {best['saved']}")
    del net, optimizer, scaler, loader, ft_ds, ft_cache
    gc.collect()
    torch.cuda.empty_cache() if device.type == "cuda" else None


Validation for fine-tuning: 48 val labels with damaged text
Baseline (v2) on this val subset: masked PSNR 22.86 dB | line CER 57.15 % | word acc 10.75 %
Fine-tune batch size 16 fits
RAM before fine-tuning: 12 %
OCR fine-tuning for 6.50 h | batch 16 | lambda_text 0.02 | lr 3.0e-05
  step   200 | loss 0.0846 | v 0.0031 | l1 0.0613 | text 0.683 (2.0 lines/batch) | lr 3.00e-05 | 0.12 h | RAM 15 %
  step   400 | loss 0.0857 | v 0.0035 | l1 0.0405 | text 0.628 (1.9 lines/batch) | lr 2.99e-05 | 0.22 h | RAM 16 %
  step   600 | loss 0.0654 | v 0.0036 | l1 0.0345 | text 0.558 (2.1 lines/batch) | lr 2.98e-05 | 0.31 h | RAM 18 %
  step   800 | loss 0.0480 | v 0.0019 | l1 0.0464 | text 0.554 (2.0 lines/batch) | lr 2.97e-05 | 0.39 h | RAM 20 %
  step  1000 | loss 0.1098 | v 0.0046 | l1 0.0383 | text 0.506 (2.2 lines/batch) | lr 2.96e-05 | 0.46 h | RAM 21 %
  step  1200 | loss 0.1339 | v 0.0055 | l1 0.0397 | text 0.570 (2.1 lines/batch) | lr 2.95e-05 | 0.52 h | RAM 23 %
  step  1400 | loss 0.1173 | 

In [12]:
# Cell 12: TEST evaluation - image metrics + text accuracy on the damaged lines, with / without OCR re-ranking
import pandas as pd
ft_model = None
if FT_PATH.exists():
    ft_model = copy.deepcopy(baseline)
    ft_model.load_state_dict(torch.load(FT_PATH, map_location=device, weights_only=False)["ema"])
    ft_model.eval()
    print("Loaded OCR-fine-tuned EMA from", FT_PATH)
else:
    print("No OCR-fine-tuned checkpoint (fine-tuning skipped or never beat the baseline) -> baseline rows only")
ft_net = (nn.DataParallel(ft_model) if (ft_model is not None and NUM_GPUS > 1) else ft_model)

N_RR = CFG["rerank_n"]
models_eval = {"v2": base_net} if ft_net is None else {"v2": base_net, "v3_ocr_ft": ft_net}
test_ocr_idx = text_hit_indices(test_samples, CFG["ocr_test_images"])

# time one batch and shrink the subset if needed
_t = time.time()
_c, _d, _m, _ = next(iter(DataLoader(IndexedSubset(test_ds, test_ocr_idx[:CFG["eval_batch_size"]]), batch_size=CFG["eval_batch_size"])))
for _net in models_eval.values():
    sample_candidates(_net, _d.to(device), _m.to(device), N_RR, CFG["ddim_steps"])
sec_per_img = (time.time() - _t) / _c.shape[0] * 1.4     # +40 % for OCR / metrics
cap = int(max(hours_left() - 0.3, 0) * 3600 / sec_per_img)
if cap < len(test_ocr_idx):
    test_ocr_idx = [test_ocr_idx[j] for j in np.linspace(0, len(test_ocr_idx) - 1, max(cap, 1)).round().astype(int)]
print(f"~{sec_per_img:.1f} s per test image -> evaluating {len(test_ocr_idx)} test labels whose damage hits text "
      f"| {hours_left():.2f} h left")

methods = ["damaged_input", "opencv_telea"] + [f"{k}{s}" for k in models_eval for s in ("", f"+rerank{N_RR}")]
img_rows = {k: [] for k in methods}
txt_rows = {k: [] for k in methods}
examples = []
loader = DataLoader(IndexedSubset(test_ds, test_ocr_idx), batch_size=CFG["eval_batch_size"], num_workers=CFG["num_workers"])
t0, done = time.time(), 0
for c, d, m, idx in loader:
    if hours_left() < 0.2:
        print(f"Time guard: stopping after {done} images"); break
    dd, md = d.to(device), m.to(device)
    det = [easy_detect_lines(to_np((d[i] + 1) / 2)) for i in range(d.shape[0])]   # boxes from the DAMAGED input
    outs = {}
    for name, netw in models_eval.items():
        cands = sample_candidates(netw, dd, md, N_RR, CFG["ddim_steps"])
        outs[name] = cands[0]
        outs[f"{name}+rerank{N_RR}"], _ = rerank(cands, dd, md, det)
    for i in range(c.shape[0]):
        mm = m[i, 0].numpy() > 0.5
        clean_i, known_i = to_np((c[i] + 1) / 2), to_np((d[i] + 1) / 2)
        cf = test_samples[int(idx[i])]["clean"]
        imgs = {"damaged_input": known_i, "opencv_telea": telea_baseline(known_i, mm)}
        for k, v in outs.items():
            imgs[k] = harmonise(to_np(v[i]), known_i, mm)
        for k, im in imgs.items():
            r = {"masked_psnr": psnr(im[mm], clean_i[mm])}
            ys, xs = np.nonzero(mm)
            if ys.max() - ys.min() >= 7 and xs.max() - xs.min() >= 7:
                y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
                r["masked_bbox_ssim"] = structural_similarity(im[y0:y1, x0:x1], clean_i[y0:y1, x0:x1], channel_axis=2, data_range=1.0)
            img_rows[k].append(r)
            txt_rows[k].append(text_scores(im, clean_i, cf, mm))
        if len(examples) < 6 and txt_rows["damaged_input"][-1]:
            examples.append((clean_i, imgs, mm, cf))
    done += c.shape[0]
    del outs, cands; gc.collect()
    if done % (CFG["eval_batch_size"] * 5) == 0:
        print(f"  {done}/{len(test_ocr_idx)} | {(time.time()-t0)/60:.1f} min | RAM {ram_percent():.0f} %")

# ---------- summary ----------
def text_summary(name):
    lines = [r for img in txt_rows[name] for r in img]
    dmg = [r for img in txt_rows["damaged_input"] for r in img]
    broken = [j for j, r in enumerate(dmg) if r["exact"] < 1]          # lines the damage made unreadable
    fixed = np.mean([lines[j]["exact"] for j in broken]) * 100 if broken and len(lines) == len(dmg) else float("nan")
    return {"line_cer": 100 * np.mean([r["cer"] for r in lines]), "word_acc": 100 * np.mean([r["exact"] for r in lines]),
            "fixed": fixed, "n_lines": len(lines)}

summary = {}
for k in methods:
    s = text_summary(k)
    s["masked_psnr"] = np.mean([r["masked_psnr"] for r in img_rows[k]])
    s["masked_bbox_ssim"] = np.nanmean([r.get("masked_bbox_ssim", np.nan) for r in img_rows[k]])
    summary[k] = s
hdr = f"{'method':22s} {'line CER %':>10s} {'word acc %':>10s} {'fixed %':>8s} {'mPSNR':>7s} {'mSSIM':>7s}"
print(f"\nTEST: {done} labels, {summary['damaged_input']['n_lines']} damaged text lines (DDIM-{CFG['ddim_steps']}, {best_h})")
print(hdr); print("-" * len(hdr))
for k, s in summary.items():
    print(f"{k:22s} {s['line_cer']:10.2f} {s['word_acc']:10.2f} {s['fixed']:8.2f} {s['masked_psnr']:7.2f} {s['masked_bbox_ssim']:7.4f}")
print("line CER / word acc: EasyOCR reading of each damaged line vs EasyOCR reading of the same line on the clean label")
print("fixed %: of the lines the damage made unreadable (damaged input read wrong), % read exactly right after reconstruction")
pd.DataFrame([{"method": k, **v} for k, v in summary.items()]).to_csv(OUT_DIR / "test_text_metrics_v3.csv", index=False)
with open(OUT_DIR / "test_text_lines_v3.json", "w") as fh:
    json.dump({k: [[{kk: (vv if kk != "box" else list(map(int, vv))) for kk, vv in r.items()} for r in img] for img in v]
               for k, v in txt_rows.items()}, fh)


Loaded OCR-fine-tuned EMA from /kaggle/working/best_ema_ocr.pth
~22.6 s per test image -> evaluating 300 test labels whose damage hits text | 7.04 h left
  80/300 | 21.9 min | RAM 98 %


In [ ]:
# Cell 13: what OCR reads on the damaged lines (zoomed on each damaged label region)
show = [k for k in ["damaged_input", "opencv_telea", "v2", f"v2+rerank{N_RR}", "v3_ocr_ft", f"v3_ocr_ft+rerank{N_RR}"] if k in methods]
for n, (clean_i, imgs, mm, cf) in enumerate(examples[:4]):
    ents = damaged_entries(ocr_cache.get(cf, []), mm)
    xs0 = min(e["box"][0] for e in ents); ys0 = min(e["box"][1] for e in ents)
    xs1 = max(e["box"][2] for e in ents); ys1 = max(e["box"][3] for e in ents)
    ys, xs = np.nonzero(mm)
    y0, y1 = max(min(ys0, ys.min()) - 10, 0), min(max(ys1, ys.max()) + 10, mm.shape[0])
    x0, x1 = max(min(xs0, xs.min()) - 10, 0), min(max(xs1, xs.max()) + 10, mm.shape[1])
    panels = [(k, imgs[k]) for k in show] + [("ground truth", clean_i)]
    fig, ax = plt.subplots(1, len(panels), figsize=(3.6 * len(panels), 3.8))
    for a, (k, im) in zip(ax, panels):
        a.imshow(np.clip(im[y0:y1, x0:x1], 0, 1))
        if k == "damaged_input":
            a.contour(mm[y0:y1, x0:x1], levels=[0.5], colors="r", linewidths=0.8)
        reads = easy_read_boxes(im, [e["box"] for e in ents])
        a.set_title(k + "\n" + "\n".join(f"'{t[:22]}'" for t in reads[:3]), fontsize=8)
        a.axis("off")
    plt.tight_layout(); plt.savefig(OUT_DIR / f"ocr_example_{n}.png", dpi=110); plt.show()


In [ ]:
# Cell 14: fine-tuning curve and output files
if ft_history:
    h = ft_history
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    ax[0].plot([r["hours"] for r in h], [100 * r["val_cer"] for r in h], "o-"); ax[0].axhline(100 * base_val[1], ls="--", c="gray", label="v2 baseline")
    ax[0].set_xlabel("fine-tuning hours"); ax[0].set_ylabel("val line CER (%)"); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot([r["hours"] for r in h], [r["val_masked_psnr"] for r in h], "o-", c="tab:green"); ax[1].axhline(base_val[0], ls="--", c="gray")
    ax[1].set_xlabel("fine-tuning hours"); ax[1].set_ylabel("val masked PSNR (dB)"); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR / "ft_curves.png", dpi=120); plt.show()
print(f"Notebook wall time: {hours_used():.2f} h")
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:34s} {p.stat().st_size/2**20:8.2f} MB")


In [ ]:
# Cell 15: save 15 test-set reconstruction examples (full label + zoom on the damage) as PNGs
N_EX = 15
VIS_DIR = OUT_DIR / "test_examples"
VIS_DIR.mkdir(exist_ok=True)

if hours_used() > 11.3:                                   # real Kaggle limit is 12 h
    print("Not enough time left for the example figures; skipping")
else:
    rng_ex = np.random.default_rng(7)                     # fixed seed -> same 15 labels every run
    pool_idx = test_ocr_idx if len(test_ocr_idx) >= N_EX else list(range(len(test_ds)))
    ex_idx = sorted(rng_ex.choice(pool_idx, size=min(N_EX, len(pool_idx)), replace=False).tolist())
    ex_loader = DataLoader(IndexedSubset(test_ds, ex_idx), batch_size=5, num_workers=0)
    order = [k for k in ["v2", f"v2+rerank{N_RR}", "v3_ocr_ft", f"v3_ocr_ft+rerank{N_RR}"]]
    n_saved = 0
    for c, d, m, idx in ex_loader:
        dd, md = d.to(device), m.to(device)
        det = [easy_detect_lines(to_np((d[i] + 1) / 2)) for i in range(d.shape[0])]
        outs = {}
        for name, netw in models_eval.items():
            cands = sample_candidates(netw, dd, md, N_RR, CFG["ddim_steps"])
            outs[name] = cands[0]
            outs[f"{name}+rerank{N_RR}"], _ = rerank(cands, dd, md, det)
        for i in range(c.shape[0]):
            mm = m[i, 0].numpy() > 0.5
            clean_i, known_i = to_np((c[i] + 1) / 2), to_np((d[i] + 1) / 2)
            panels = [("damaged input", known_i), ("OpenCV Telea", telea_baseline(known_i, mm))]
            panels += [(k, harmonise(to_np(outs[k][i]), known_i, mm)) for k in order if k in outs]
            panels += [("ground truth", clean_i)]
            if mm.any():
                ys, xs = np.nonzero(mm); pad = 16
                y0, y1 = max(ys.min() - pad, 0), min(ys.max() + pad, mm.shape[0])
                x0, x1 = max(xs.min() - pad, 0), min(xs.max() + pad, mm.shape[1])
            else:
                y0, y1, x0, x1 = 0, mm.shape[0], 0, mm.shape[1]
            fig, ax = plt.subplots(2, len(panels), figsize=(3.2 * len(panels), 6.8))
            for j, (title, im) in enumerate(panels):
                sub = f"\nmPSNR {psnr(im[mm], clean_i[mm]):.1f} dB" if (mm.any() and title != "ground truth") else ""
                ax[0, j].imshow(np.clip(im, 0, 1)); ax[0, j].set_title(title + sub, fontsize=9)
                ax[1, j].imshow(np.clip(im[y0:y1, x0:x1], 0, 1)); ax[1, j].set_title("zoom", fontsize=8)
                if title == "damaged input":
                    ax[0, j].contour(mm, levels=[0.5], colors="r", linewidths=0.7)
                    ax[1, j].contour(mm[y0:y1, x0:x1], levels=[0.5], colors="r", linewidths=0.7)
            for a in ax.ravel():
                a.axis("off")
            n_saved += 1
            fname = Path(test_samples[int(idx[i])]["damaged"]).stem
            fig.suptitle(f"Test example {n_saved}/{len(ex_idx)}: {fname}  (red = damage mask)", fontsize=11)
            plt.tight_layout()
            plt.savefig(VIS_DIR / f"example_{n_saved:02d}_{fname}.png", dpi=110)
            if n_saved <= 3:
                plt.show()                                # show the first 3 inline, save all 15
            plt.close(fig)
    shutil.make_archive(str(OUT_DIR / "test_examples"), "zip", VIS_DIR)
    print(f"Saved {n_saved} example figures to {VIS_DIR} (+ test_examples.zip for one-click download)")